In [1]:


import pandas as pd
import numpy as np
import re
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

df_train = pd.read_csv("../data/train.csv")
df_val   = pd.read_csv("../data/val.csv")
df_test  = pd.read_csv("../data/test.csv")

In [2]:
# nettoyage texte 
def clean_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()                          # minuscules
    text = re.sub(r"http\S+", " ", text)         # supprimer URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)    # garder lettres/chiffres
    text = re.sub(r"\s+", " ", text).strip()     # espaces multiples
    return text

df_train["clean_text"] = df_train["text"].apply(clean_text)
df_val["clean_text"]   = df_val["text"].apply(clean_text)
df_test["clean_text"]  = df_test["text"].apply(clean_text)

print("Exemple avant :", df_train["text"].iloc[0][:80])
print("Exemple après :", df_train["clean_text"].iloc[0][:80])

Exemple avant : What are the best practices for secure coding in Python?
Exemple après : what are the best practices for secure coding in python


In [4]:
# Features manuelles
KEYWORDS = ["ignore", "bypass", "jailbreak", "forget", "pretend",
            "roleplay", "override", "disregard", "base64", "system"]

def manual_features(df):
    feats = pd.DataFrame()
    feats["text_len"]      = df["text"].str.len()
    feats["word_count"]    = df["text"].str.split().str.len()
    feats["has_keyword"]   = df["text"].str.lower().apply(
                                lambda x: int(any(k in str(x) for k in KEYWORDS)))
    feats["has_base64"]    = df["text"].str.contains(
                                r"[A-Za-z0-9+/]{20,}={0,2}", regex=True).astype(int)
    feats["uppercase_ratio"] = df["text"].apply(
                                lambda x: sum(1 for c in str(x) if c.isupper()) / max(len(str(x)),1))
    return feats

feats_train = manual_features(df_train)
feats_val   = manual_features(df_val)
feats_test  = manual_features(df_test)
print(feats_train.head())

   text_len  word_count  has_keyword  has_base64  uppercase_ratio
0        56          10            0           0         0.035714
1        68          13            0           0         0.029412
2        36           5            0           0         0.027778
3        65          10            0           0         0.015385
4        58          11            1           0         0.068966


In [5]:
#  TF-IDF Vectorizer 
tfidf = TfidfVectorizer(
    max_features=10000,
    ngram_range=(1, 2),       # unigrams + bigrams
    sublinear_tf=True,
    min_df=2
)

X_tfidf_train = tfidf.fit_transform(df_train["clean_text"])
X_tfidf_val   = tfidf.transform(df_val["clean_text"])
X_tfidf_test  = tfidf.transform(df_test["clean_text"])

print("Shape TF-IDF train:", X_tfidf_train.shape)

Shape TF-IDF train: (4391, 9988)


In [6]:
# Combiner TF-IDF + features manuelles 
from scipy.sparse import hstack, csr_matrix

X_train = hstack([X_tfidf_train, csr_matrix(feats_train.values)])
X_val   = hstack([X_tfidf_val,   csr_matrix(feats_val.values)])
X_test  = hstack([X_tfidf_test,  csr_matrix(feats_test.values)])

y_train = df_train["label"].values
y_val   = df_val["label"].values
y_test  = df_test["label"].values

print("X_train final shape:", X_train.shape)
print("Classes :", np.unique(y_train))

# Sauvegarder le vectorizer
joblib.dump(tfidf, "../models/tfidf.pkl")
print("TF-IDF sauvegardé dans models/tfidf.pkl")

X_train final shape: (4391, 9993)
Classes : [0 1]
TF-IDF sauvegardé dans models/tfidf.pkl
